In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import joblib

# File Paths
X_CSV = "../data stuff/augmented_telemetry_filtered_with_location.csv"
Y_CSV = "../data stuff/next24h_consumption.csv"
OUT_CSV = "../data stuff/predicted_24h_consumption_per_location.csv"
FINAL_REPORT_CSV = "../data stuff/final_comparison_report.csv"

def run_demand_prediction():
    df_x = pd.read_csv(X_CSV)
    df_y = pd.read_csv(Y_CSV)

    le = LabelEncoder()
    df_x['location_encoded'] = le.fit_transform(df_x['location_id'])

    df_x['recorded_at'] = pd.to_datetime(df_x['recorded_at'])
    df_x['day_of_week'] = df_x['recorded_at'].dt.dayofweek
    
    train_agg = df_x.groupby(['location_id', 'date']).agg({
        'dispensed_l': 'sum',
        'pressure_pa': 'mean',
        'device_id': 'nunique',
        'day_of_week': 'first',
        'location_encoded': 'first'
    }).reset_index()

    train_agg = train_agg.sort_values(['location_id', 'date'])
    train_agg['prev_day_l'] = train_agg.groupby('location_id')['dispensed_l'].shift(1)
    train_agg['prev_day_l'] = train_agg['prev_day_l'].fillna(train_agg.groupby('location_id')['dispensed_l'].transform('mean'))

    features = ['location_encoded', 'prev_day_l', 'pressure_pa', 'device_id', 'day_of_week']
    X_train = train_agg[features]
    y_train = train_agg['dispensed_l']

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    joblib.dump(model, 'water_demand_model.pkl')
    joblib.dump(features, 'model_features.pkl')
    joblib.dump(le, 'location_encoder.pkl')
    print(f"Model assets saved: 'water_demand_model.pkl', 'model_features.pkl', and 'location_encoder.pkl'")

    dev_counts = df_x.groupby('location_id')['device_id'].nunique().to_dict()
    last_day_consumption = train_agg.sort_values('date').groupby('location_id').last()['dispensed_l'].to_dict()
    
    latest_snapshots = df_x.sort_values('recorded_at').groupby('location_id').last().reset_index()
    latest_snapshots['device_id'] = latest_snapshots['location_id'].map(dev_counts)
    latest_snapshots['prev_day_l'] = latest_snapshots['location_id'].map(last_day_consumption)
    latest_snapshots['location_encoded'] = le.transform(latest_snapshots['location_id'])
    
    predict_df = latest_snapshots[latest_snapshots['location_id'].isin(df_y['location_id'])].copy()
    X_test = predict_df[features]
    
    predict_df['model_forecast_l'] = model.predict(X_test)
    
    final_comparison = df_y.merge(
        predict_df[['location_id', 'model_forecast_l']], 
        on='location_id', 
        how='left'
    )


    actual = final_comparison['predicted_demand_l']
    predicted = final_comparison['model_forecast_l']
    print("\n MODEL PERFORMANCE")
    print(f"MSE: {round(mean_squared_error(actual, predicted), 2)}")
    print(f"MAE: {round(mean_absolute_error(actual, predicted), 2)} Liters")

    final_comparison.to_csv(OUT_CSV, index=False)
    return final_comparison

def generate_comparison_report(df_results):
    df = df_results.rename(columns={
        'predicted_demand_l': 'actual_avg_demand_l',
        'model_forecast_l': 'ml_predicted_demand_l'
    })

    # Metrics
    df['variance_l'] = df['ml_predicted_demand_l'] - df['actual_avg_demand_l']
    df['accuracy_pct'] = (1 - (abs(df['variance_l']) / df['actual_avg_demand_l'])) * 100

    q_high = df['ml_predicted_demand_l'].quantile(0.75)
    df['priority_level'] = df['ml_predicted_demand_l'].apply(
        lambda x: 'CRITICAL' if x > q_high else 'NORMAL'
    )

    df = df.round({'actual_avg_demand_l': 1, 'ml_predicted_demand_l': 1, 'variance_l': 1, 'accuracy_pct': 1})

    cols = ['location_id', 'actual_avg_demand_l', 'ml_predicted_demand_l', 'priority_level']
    df.to_csv(FINAL_REPORT_CSV, index=False)
    
    print(f"Final Comparison Report saved to: {FINAL_REPORT_CSV}")
    print("\nTop 5 Predictions vs Actuals")
    print(df[cols].head())

if __name__ == "__main__":
    results_df = run_demand_prediction()
    generate_comparison_report(results_df)

Model assets saved: 'water_demand_model.pkl', 'model_features.pkl', and 'location_encoder.pkl'

 MODEL PERFORMANCE
MSE: 79926.91
MAE: 232.35 Liters
Final Comparison Report saved to: ../data stuff/final_comparison_report.csv

Top 5 Predictions vs Actuals
  location_id  actual_avg_demand_l  ml_predicted_demand_l priority_level
0     loc_001               2965.0                 2720.8         NORMAL
1     loc_002               2812.8                 2688.8         NORMAL
2     loc_003               6364.5                 6607.9       CRITICAL
3     loc_004               3104.1                 3048.1         NORMAL
4     loc_005               3617.9                 3594.5       CRITICAL


In [14]:
import joblib
import pandas as pd

def test_inference(location_id, prev_day_l, pressure, device_count, day_idx):
    # 1. Load Assets
    model = joblib.load('water_demand_model.pkl')
    features = joblib.load('model_features.pkl')
    le = joblib.load('location_encoder.pkl')
    
    # 2. Convert string location to the number the model learned
    try:
        loc_numeric = le.transform([location_id])[0]
    except ValueError:
        print(f"Warning: {location_id} is a new location. Using default encoding.")
        loc_numeric = 0 

    # 3. Predict
    input_df = pd.DataFrame([{
        'location_encoded': loc_numeric,
        'prev_day_l': prev_day_l,
        'pressure_pa': pressure,
        'device_id': device_count,
        'day_of_week': day_idx
    }])
    
    prediction = model.predict(input_df[features])[0]
    return round(prediction, 2)

# --- TEST CASE ---
location = "loc_019"
res = test_inference(location, 3200.0, 4500.0, 10, 0)
print(f"Prediction for {location}: {res} Liters")

Prediction for loc_019: 3202.14 Liters
